# Seminar Notebook 3.4: Classification

**LSE MY459: Computational Text Analysis and Large Language Models** (WT 2026)

**Ryan Hübert**

This notebook covers supervised classification of texts using Naive Bayes, regularised regression, and cross-validation.

## Set up steps

### Directory management

We begin with some directory management to specify the file path to the folder on your computer where you wish to store data for this notebook.

In [ ]:
import os
sdir = os.path.join(os.path.expanduser("~"), "LSE-MY459-WT26", "SeminarWeek07")
dfm_path = os.path.join(sdir, "candidate-dfm.npz")
vocab_path = os.path.join(sdir, "candidate-dfm-features.txt")
corpus_path = os.path.join(sdir, "candidate-corpus.csv")
if any(not os.path.exists(x) for x in [dfm_path, vocab_path, corpus_path]):
    raise FileNotFoundError("You must run the first notebook before proceeding!")

### Loading the data

We will use the corpus of tweets posted by the major candidates for the 2016 U.S. presidential election, for which we created a DFM in `01-making-dfm.ipynb`. 

In [ ]:
import pandas as pd
import numpy as np
from scipy import sparse

# Load the sparse DFM
dfm = sparse.load_npz(dfm_path)
print(f"DFM shape: {dfm.shape}")

# Load the vocabulary
with open(vocab_path, "r") as f:
    vocabulary = f.read().split("\n")
print(f"Number of features: {len(vocabulary)}")

# Load the corpus metadata
corpus = pd.read_csv(corpus_path)
corpus.head()

## Labelling and splitting data

Our goal is to build a classifier that can predict whether tweets were posted by Hillary Clinton or not. This is a binary classification task, so we need to create a (binary) variable indicating whether each tweet was posted by Hillary Clinton. 

In [ ]:
# Create Clinton label based on screen_name
corpus["Clinton"] = corpus["screen_name"].apply(lambda x: 1 if "Clinton" in x else 0)
print(corpus["Clinton"].value_counts())

As discussed in lecture, we need to split our data into training and test sets. The training set is used to build the model, and the test set is held out to evaluate performance on unseen data. This helps us detect overfitting. We will use 75% for training and 25% for test.

In [ ]:
from sklearn.model_selection import train_test_split

dfm_train, dfm_test, labels_train, labels_test = train_test_split(
    dfm, corpus["Clinton"],
    test_size=0.25,
    random_state=755
)

In [ ]:
print(f"Training labels: {pd.Series(labels_train).value_counts().to_dict()}")
print(f"Test labels: {pd.Series(labels_test).value_counts().to_dict()}")

## Classifying with Naive Bayes

Our first classifier is Naive Bayes. This model uses Bayes' theorem to calculate the probability of a document belonging to each class, assuming that words are conditionally independent given the class. For text with word counts, we use the Multinomial Naive Bayes model.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Train Naive Bayes model on training set
nb = MultinomialNB(alpha=1.0)  # alpha is the Laplace smoothing parameter
nb.fit(dfm_train, labels_train)

Using the estimated model, which we called `nb`, let's do prediction! First, we can extract the predicted probabilities.

In [ ]:
nb_probs_insample = pd.DataFrame(nb.predict_proba(dfm_train), columns=nb.classes_)
nb_probs_test = pd.DataFrame(nb.predict_proba(dfm_test), columns=nb.classes_)

print("Predicted probabilities in-sample (first five tweets)")
print(nb_probs_insample.head())
print("Predicted probabilities for the test set (first five tweets)")
print(nb_probs_test.head())

Now we predict labels for both the training set (in-sample) and the test set (out-of-sample).

In [ ]:
# Predict in-sample and out-of-sample
preds_insample = nb.predict(dfm_train)
preds_test = nb.predict(dfm_test)

### Confusion matrices

We create confusion matrices to see how well our classifier performed. The rows represent true labels and columns represent predicted labels.

In [ ]:
# Create confusion matrices
cm_insample = pd.crosstab(labels_train, preds_insample, rownames=["True"], colnames=["Predicted"])
cm_test = pd.crosstab(labels_test, preds_test, rownames=["True"], colnames=["Predicted"])

print("In-sample confusion matrix:")
print(cm_insample)
print()
print("Out-of-sample confusion matrix:")
print(cm_test)

### Performance metrics

Let's create a function that calculates precision, recall, F1 and accuracy for a specific class. This helps us understand how well the classifier performs for each category.

In [ ]:
def performance_stats_k(cm, k, insample=True):
    # Convert to numpy array for easier indexing
    cm_arr = cm.values
    classes = cm.index.tolist()
    i = classes.index(k)
    
    # Correct classifications for class k
    correct = cm_arr[i, i]

    # Marginal totals
    pred_k = cm_arr[:, i].sum() # column marginal (predicted as k)
    true_k = cm_arr[i, :].sum() # row marginal (true k)
    
    # Precision: correct / total predicted as k
    precision = correct / pred_k if pred_k > 0 else 0
    
    # Recall: correct / total truly in k
    recall = correct / true_k if true_k > 0 else 0
    
    # F1: harmonic mean of precision and recall
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Accuracy: total correct / total
    accuracy = np.diag(cm_arr).sum() / cm_arr.sum()
    
    sample_type = "In-sample" if insample else "Out-of-sample"
    print("Confusion matrix:")
    print(cm)
    print(f"\n{sample_type} performance")
    print(f"  accuracy = {accuracy:.3f}")
    print(f"")
    print(f"     class = {k}")
    print(f" precision = {precision:.3f}")
    print(f"    recall = {recall:.3f}")
    print(f"        F1 = {f1:.3f}")

In [ ]:
print("=" * 50)
print("NAIVE BAYES - Clinton class")
print("=" * 50)
performance_stats_k(cm_insample, 1, insample=True)
print("=" * 50)
performance_stats_k(cm_test, 1, insample=False)
print("=" * 50)

### Using sklearn's built-in metrics

The `sklearn.metrics` module provides functions to calculate these metrics automatically. This is often more convenient than computing them by hand. That said, you should know how to calculate by hand.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Using sklearn to calculate the same metrics for out-of-sample predictions
print(accuracy_score(labels_test, preds_test))
print(precision_score(labels_test, preds_test))
print(recall_score(labels_test, preds_test))
print(f1_score(labels_test, preds_test))

Conveniently, `sklearn` also has a `classification_report()` function that provides all stats at once.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(labels_train, preds_insample))
print(classification_report(labels_test, preds_test))

Notice that in-sample performance is typically better than out-of-sample performance. This is because the model has "seen" the training data and may have overfit to it.

### Tuning the smoothing parameter

Above we used `alpha=1.0` (standard Laplace smoothing), but this is a hyperparameter that can be tuned. Unlike ridge and lasso, sklearn doesn't provide a `MultinomialNBCV` function, so we use `GridSearchCV` to search over candidate values.

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold

# Create folds for cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=846)

# Search over smoothing parameters
alphas = np.logspace(-3, 1, 20)

nb_tuned = GridSearchCV(
    MultinomialNB(),
    param_grid={'alpha': alphas},
    cv=cv,
    scoring='f1'
)
nb_tuned.fit(dfm_train, labels_train)

print(f"Best alpha: {nb_tuned.best_params_['alpha']:.4f}")

In [ ]:
# Evaluate tuned model on test set
preds_test_tuned = nb_tuned.predict(dfm_test)

print("Tuned Naive Bayes (test set):")
print(classification_report(labels_test, preds_test_tuned))

Compare with the original results above (alpha=1.0). Tuning often provides modest improvements, though for Naive Bayes the default Laplace smoothing is frequently near-optimal.

## Classifying with regularised regression

Another common approach to classification is regularised linear modelling, such as linear regression with a ridge or lasso penalty (this is the "regularisation"). Unlike Naive Bayes, these models directly estimate the relationship between features and the outcome. Regularisation penalises large coefficients, helping to prevent overfitting. For text data, it is often beneficial to apply TF-IDF weighting and/or normalisation. TF-IDF reduces the influence of very common words, while normalisation ensures that longer documents do not dominate the model. These transformations typically improve performance, though the optimal choice depends on the data.

Why didn't we use TF-IDF for Naive Bayes? Multinomial Naive Bayes has its own generative model for word counts, so it expects raw counts as input. Linear regression has no such built-in model for text, so rescaling the features with TF-IDF helps it work better.

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf = TfidfTransformer()
dfm_train_tfidf = tfidf.fit_transform(dfm_train)
dfm_test_tfidf = tfidf.transform(dfm_test)

Let's start with a ridge regression on our DFM. Recall that the optimal penalty ($\lambda$) should be chosen by cross-validation. In `sklearn`, `RidgeCV()` allows us to estimate a linear regression model with the ridge (L2) penalty, where the penalty is chosen via cross-validation. You first need to define the folds for cross-validation, and the values of $\lambda$ you want the algorithm to try. We'll do five fold cross-validation and search across 20 different values of $\lambda$.

In [ ]:
from sklearn.model_selection import KFold

# Create folds for finding hyperparameter lambda
cv = KFold(n_splits=5, shuffle=True, random_state=846)

# Which values of lambda should we try?
lambdas = np.logspace(-4, 4, 20)
print([f"{x:.2f}" for x in lambdas])

In [ ]:
from sklearn.linear_model import RidgeCV

# Ridge regression with cross-validation
ridge = RidgeCV(
    alphas=lambdas,  # Note sklearn calls the hyperparameter alpha, not lambda
    cv=cv
)

# Fit (labels must be numeric, e.g. 0/1)
ridge.fit(dfm_train_tfidf, labels_train)

What was the best penalty that the algorithm found?

In [ ]:
ridge.alpha_ # this is the best lambda identified by CV

Using the estimated model, which we called `ridge`, let's start doing prediction. First, we can extract the continuous predictions from the model, which we call "scores."

In [ ]:
# Continuous predictions
ridge_scores_insample = ridge.predict(dfm_train_tfidf)
ridge_scores_test = ridge.predict(dfm_test_tfidf)

print("Predicted scores in-sample (first five tweets)")
print(ridge_scores_insample[:5])

print("Predicted scores for the test set (first five tweets)")
print(ridge_scores_test[:5])

Usually we want to predict the _label_ for each document. We'll predict the labels for each document based on whether the score is greater than 0.5. We did not cover this in lecture, but the choice of the threshold could be optimised. Here, we use 0.5, but you can choose the threshold that optimises the performance statistics. You can learn more about this in a machine learning course.

In [ ]:
# Convert to class predictions (using threshold = 0.5)
ridge_preds_insample = (ridge_scores_insample >= 0.5).astype(int)
ridge_preds_test = (ridge_scores_test >= 0.5).astype(int)

print("Predicted labels in-sample (first five tweets)")
print(ridge_preds_insample[0:5])
print("Predicted labels for the test set (first five tweets)")
print(ridge_preds_test[0:5])

Let's now examine the performance stats for this ridge regression.

In [ ]:
# Create confusion matrices
ridge_cm_insample = pd.crosstab(labels_train, ridge_preds_insample, rownames=["True"], colnames=["Predicted"])
ridge_cm_test = pd.crosstab(labels_test, ridge_preds_test, rownames=["True"], colnames=["Predicted"])

print("=" * 50)
print("RIDGE REGRESSION - Clinton class")
print("=" * 50)
performance_stats_k(ridge_cm_insample, 1, insample=True)
print("=" * 50)
performance_stats_k(ridge_cm_test, 1, insample=False)
print("=" * 50)

### Other penalties 

As discussed in lecture, the ridge (L2) penalty is not the only option. An alternative is the lasso (L1) penalty. Unlike ridge, lasso can shrink some coefficients exactly to zero, effectively performing feature selection. A more general approach is the elastic net, which combines L1 and L2 penalties. In `sklearn`, `ElasticNetCV()` can be used to estimate lasso or elastic net with cross-validation. In the example below, we use the lasso penalty, but this can be extended to elastic net by choosing an `l1_ratio` between 0 and 1. 

Note: when I first ran this, I used `np.logspace(-4, 1, 100)` for the lambda grid, but the best lambda was at the boundary (the smallest value). This means the cross-validation didn't find an interior optimum, so I widened the grid to `np.logspace(-6, 1, 100)` to give the algorithm more room to search.

In [ ]:
from sklearn.linear_model import ElasticNetCV

# Which values of lambda should we try?
lambdas = np.logspace(-6, 1, 100)

# Lasso regression with cross-validation
lasso = ElasticNetCV(
    l1_ratio=1.0,     # L1/Lasso penalty 
    alphas=lambdas,   # lambda values to try
    cv=cv,            # 5-fold cross-validation
    n_jobs=None,      # Use 1 CPU -- if you want mutlicore set -1 (all) or a specific number like 2
    random_state=846
)

# Fit the lasso
lasso.fit(dfm_train_tfidf, labels_train)

# This is the best lambda identified by CV
lasso.alpha_ 

Now we extract both the scores and the predicted labels, just as we did above.

In [ ]:
# Continuous predictions
lasso_scores_insample = lasso.predict(dfm_train_tfidf)
lasso_scores_test = lasso.predict(dfm_test_tfidf)

# Convert to class predictions (threshold = 0.5)
lasso_preds_insample = (lasso_scores_insample >= 0.5).astype(int)
lasso_preds_test = (lasso_scores_test >= 0.5).astype(int)

In [ ]:
# Create confusion matrices
lasso_cm_insample = pd.crosstab(labels_train, lasso_preds_insample, rownames=["True"], colnames=["Predicted"])
lasso_cm_test = pd.crosstab(labels_test, lasso_preds_test, rownames=["True"], colnames=["Predicted"])

print("=" * 50)
print("LASSO REGRESSION - Clinton class")
print("=" * 50)
performance_stats_k(lasso_cm_insample, 1, insample=True)
print("=" * 50)
performance_stats_k(lasso_cm_test, 1, insample=False)
print("=" * 50)

With lasso, some coefficients are set exactly to zero. Let's see how many features remain.

In [ ]:
nonzero = (lasso.coef_ != 0)
zero = (lasso.coef_ == 0)

print(f"Non-zero coefficients: {np.sum(nonzero)}")
print(f"Zero coefficients: {np.sum(zero)}")

In [ ]:
pd.DataFrame({"term": [vocabulary[x] for x in range(0,len(lasso.coef_)) if lasso.coef_[x] > 0], 
              "coef": [lasso.coef_[x] for x in range(0,len(lasso.coef_)) if lasso.coef_[x] > 0]}
              ).sort_values("coef", ascending=False).iloc[0:10,]

## Robust evaluation of classifiers with cross-validation

The single train/test split we used above is simple but has a limitation: our results depend on which documents happen to end up in each set. With a different random split, we might get different performance estimates. Cross-validation addresses this by repeatedly splitting the data, training on each split, and averaging results. This gives us more robust estimates of out-of-sample performance. Note: we already used cross-validation to select optimal hyperparameters for our regularised regression models. This was a slightly different use of cross-validation---i.e., using for _tuning_ the model, not evaluating it on held-out data.

In this example, we'll use a cross-validation approach with Naive Bayes classification, but this could also be done for regularised regression.

In [ ]:
from sklearn.model_selection import cross_val_predict

# Create folds for held-out performance
cv_eval = KFold(n_splits=10, shuffle=True, random_state=7466) # use different random state from above!

# Get the labels from the entire dataset (not just training or test)
labels = corpus["Clinton"].values

# Get cross-validated predictions for the NB classifier
# Each document gets a prediction from a model trained WITHOUT that document
nb_cv_preds = cross_val_predict(MultinomialNB(alpha=1.0), dfm, labels, cv=cv_eval)

The `cross_val_predict` function above performs 10-fold cross-validation and returns predictions for every document. Crucially, each document's prediction comes from a model that was trained *without* that document. This gives us truly out-of-sample predictions for the entire dataset.

In [ ]:
# Create confusion matrices from CV predictions
nb_cv_cm = pd.crosstab(labels, nb_cv_preds, rownames=["True"], colnames=["Predicted"])

print("NAIVE BAYES (10-fold CV):")
print(nb_cv_cm)

Let’s compare the performance statistics from 10-fold cross-validation and our original 25% test set. Cross-validation often yields slightly better performance estimates because each model is trained on a larger portion of the data and the results are averaged over multiple splits, reducing variability. In contrast, a single held-out test set reflects performance on one particular split and can therefore be more variable. However, there is no guarantee that cross-validation will produce better performance statistics than a test set. Moreover, because cross-validation is typically used for model selection, its performance estimates can be slightly optimistic. The held-out test set remains the most reliable measure of out-of-sample performance.

In [ ]:
print("Naive Bayes, 10-fold CV:")
print(classification_report(labels, nb_cv_preds))

print("Naive Bayes, 25% Test Set:")
print(classification_report(labels_test, preds_test))

## Comparing classifiers

Let's compare all three classifiers side by side using the out-of-sample (test set) performance metrics for the Clinton class.

In [ ]:
def get_metrics(true, pred, name):
    """Calculate performance metrics for the positive class (Clinton = 1)."""
    cm = pd.crosstab(true, pred)
    cm_arr = cm.values
    accuracy = np.diag(cm_arr).sum() / cm_arr.sum()
    precision = cm_arr[1, 1] / cm_arr[:, 1].sum() if cm_arr[:, 1].sum() > 0 else 0
    recall = cm_arr[1, 1] / cm_arr[1, :].sum() if cm_arr[1, :].sum() > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return {"Classifier": name, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1": f1}

comparison = pd.DataFrame([
    get_metrics(labels_test, preds_test, "Naive Bayes"),
    get_metrics(labels_test, ridge_preds_test, "Ridge"),
    get_metrics(labels_test, lasso_preds_test, "Lasso"),
]).set_index("Classifier").round(3)

comparison

Notice the trade-offs between classifiers. Naive Bayes tends to have higher recall (it identifies more Clinton tweets) but lower precision (more false positives). Ridge and lasso have higher precision but lower recall. Which classifier is "best" depends on whether you care more about avoiding false positives (precision) or catching all true positives (recall). The F1 score provides a single summary that balances both.